# Updating a Partitioned Parquet Table — Dynamic Partition Overwrite

## Scenario
Move employee **ID 10** from Department 10 to Department 35 **without rewriting the entire dataset**.

| Step | What happens |
|------|-------------|
| 1 | Create synthetic employee dataset, write as Parquet partitioned by `dept_id` |
| 2 | Read only the two affected partitions (dept 10 and dept 35) |
| 3 | Apply the `dept_id` update in memory |
| 4 | Write the two updated partitions back with **dynamic partition overwrite** |
| 5 | Validate — only the two touched partitions were rewritten |

**Key concepts demonstrated**
- `spark.sql.sources.partitionOverwriteMode = dynamic` — overwrites only partitions present in the new data
- Partition pruning — reading only relevant partition directories
- Updating a partition column without a full table rewrite

## Setup

In [2]:
import tempfile, os
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, when, lit

spark = (
    SparkSession.builder
    .appName('PartitionUpdateDemo')
    .config('spark.sql.adaptive.enabled', 'true')
    .config('spark.sql.shuffle.partitions', '4')
    .getOrCreate()
)

# Enable dynamic partition overwrite — only partitions in the new data are replaced.
# Default ('static') would delete ALL partitions before writing.
spark.conf.set('spark.sql.sources.partitionOverwriteMode', 'dynamic')
print('partitionOverwriteMode:', spark.conf.get('spark.sql.sources.partitionOverwriteMode'))

partitionOverwriteMode: dynamic


## Step 1 — Create Synthetic Employee Dataset

20 employees spread across 4 departments (dept_id = 10, 20, 30, 35).
Employee 10 starts in **dept_id = 10**; we will move them to **dept_id = 35**.

In [3]:
from pyspark.sql.types import StructType, StructField, IntegerType, StringType, DoubleType

schema = StructType([
    StructField('employee_id', IntegerType(), False),
    StructField('name',        StringType(),  False),
    StructField('salary',      DoubleType(),  False),
    StructField('dept_id',     IntegerType(), False),
])

data = [
    # dept 10
    (10, 'Alice',    72000.0, 10),
    (11, 'Bob',      68000.0, 10),
    (12, 'Carol',    75000.0, 10),
    (13, 'Dave',     69000.0, 10),
    (14, 'Eve',      71000.0, 10),
    # dept 20
    (20, 'Frank',    80000.0, 20),
    (21, 'Grace',    82000.0, 20),
    (22, 'Hank',     78000.0, 20),
    (23, 'Iris',     85000.0, 20),
    (24, 'Jack',     77000.0, 20),
    # dept 30
    (30, 'Karen',    91000.0, 30),
    (31, 'Leo',      88000.0, 30),
    (32, 'Mia',      93000.0, 30),
    (33, 'Ned',      87000.0, 30),
    (34, 'Olivia',   95000.0, 30),
    # dept 35
    (35, 'Paul',     60000.0, 35),
    (36, 'Quinn',    62000.0, 35),
    (37, 'Rose',     58000.0, 35),
    (38, 'Sam',      63000.0, 35),
    (39, 'Tina',     61000.0, 35),
]

employee_df = spark.createDataFrame(data, schema)
employee_df.show(truncate=False)
print('Total rows:', employee_df.count())

Py4JJavaError: An error occurred while calling o48.showString.
: org.apache.spark.SparkException: Job aborted due to stage failure: Task 0 in stage 0.0 failed 1 times, most recent failure: Lost task 0.0 in stage 0.0 (TID 0) (PS-Laptop executor driver): org.apache.spark.SparkException: Python worker failed to connect back.
	at org.apache.spark.api.python.PythonWorkerFactory.createSimpleWorker(PythonWorkerFactory.scala:281)
	at org.apache.spark.api.python.PythonWorkerFactory.create(PythonWorkerFactory.scala:154)
	at org.apache.spark.SparkEnv.createPythonWorker(SparkEnv.scala:158)
	at org.apache.spark.api.python.BasePythonRunner.compute(PythonRunner.scala:309)
	at org.apache.spark.api.python.PythonRDD.compute(PythonRDD.scala:72)
	at org.apache.spark.rdd.RDD.computeOrReadCheckpoint(RDD.scala:374)
	at org.apache.spark.rdd.RDD.iterator(RDD.scala:338)
	at org.apache.spark.rdd.MapPartitionsRDD.compute(MapPartitionsRDD.scala:52)
	at org.apache.spark.rdd.RDD.computeOrReadCheckpoint(RDD.

## Step 2 — Write Initial Partitioned Parquet Table

Write the full dataset partitioned by `dept_id`.
This creates one directory per department: `dept_id=10/`, `dept_id=20/`, `dept_id=30/`, `dept_id=35/`.

In [ ]:
# Use a temp directory with forward slashes (required for PySpark on Windows)
BASE_PATH = tempfile.mkdtemp().replace('\\', '/') + '/employee_parquet'
print('Writing to:', BASE_PATH)

(
    employee_df
    .write
    .mode('overwrite')
    .partitionBy('dept_id')
    .parquet(BASE_PATH)
)

# Show partition directories created
print('\nPartition directories:')
for entry in sorted(os.listdir(BASE_PATH)):
    if entry.startswith('dept_id='):
        n_files = len([f for f in os.listdir(os.path.join(BASE_PATH, entry)) if f.endswith('.parquet')])
        print(f'  {entry}/ — {n_files} parquet file(s)')

## Step 3 — Inspect Initial State

Re-read the table and verify employee 10 is in **dept_id = 10**.

In [ ]:
# Re-read so Spark uses the on-disk partition layout (partition pruning applies on reads)
table_df = spark.read.parquet(BASE_PATH)

print('=== Full table (sorted by employee_id) ===')
table_df.orderBy('employee_id').show(25, truncate=False)

# Confirm employee 10's current department
emp10_row = table_df.filter(col('employee_id') == 10).first()
current_dept = emp10_row['dept_id']
print(f'Employee 10 ({emp10_row["name"]}) is currently in dept_id = {current_dept}')

## Step 4 — Read Only the Two Affected Partitions

We only need dept 10 (current) and dept 35 (target).
Using `.filter()` on a partition column triggers **partition pruning** —
Spark reads only those two directories, skipping dept 20 and dept 30 entirely.

> Check `explain()` below: `PartitionFilters: [dept_id IN (10, 35)]` confirms pruning.

In [ ]:
EMPLOYEE_ID   = 10
TARGET_DEPT   = 35

# Partition-pruned read — only dept_id=10 and dept_id=35 directories are opened
affected_df = (
    spark.read
         .parquet(BASE_PATH)
         .filter(col('dept_id').isin(current_dept, TARGET_DEPT))
)

print('=== Affected partitions (dept 10 + dept 35) ===')
affected_df.orderBy('dept_id', 'employee_id').show(truncate=False)

print('=== Physical plan — confirm PartitionFilters ===')
affected_df.explain()

## Step 5 — Apply the Update In Memory

For the single matching row (`employee_id = 10`), change `dept_id` from 10 → 35.
All other rows keep their existing `dept_id`.

In [ ]:
updated_df = (
    affected_df
    .withColumn(
        'dept_id',
        when(col('employee_id') == EMPLOYEE_ID, lit(TARGET_DEPT))
        .otherwise(col('dept_id'))
    )
)

print('=== Updated rows (employee 10 now shows dept_id=35) ===')
updated_df.orderBy('dept_id', 'employee_id').show(truncate=False)

# Sanity check: the only changed row is employee 10
changed = updated_df.filter(col('employee_id') == EMPLOYEE_ID).first()
print(f'Employee {EMPLOYEE_ID} dept_id after update: {changed["dept_id"]} (was {current_dept})')

## Step 6 — Write Back with Dynamic Partition Overwrite

`updated_df` contains rows for only dept 10 and dept 35.
With `partitionOverwriteMode = dynamic`, Spark replaces **only those two partition directories**.

- `dept_id=10/` — rewritten (employee 10 removed, others remain)
- `dept_id=35/` — rewritten (employee 10 added)
- `dept_id=20/` — **untouched**
- `dept_id=30/` — **untouched**

> If `partitionOverwriteMode` were `static` (the default), ALL four partitions would be deleted
> before writing and dept 20 + dept 30 data would be permanently lost.

In [ ]:
(
    updated_df
    .write
    .mode('overwrite')
    .partitionBy('dept_id')
    .parquet(BASE_PATH)
)
print('Write complete — only dept_id=10 and dept_id=35 were overwritten.')

## Step 7 — Validate the Result

Re-read the full table and confirm:
1. Employee 10 is now in **dept_id = 35**
2. Dept 20 and dept 30 data is **unchanged**
3. The original dept 10 no longer contains employee 10

In [ ]:
final_df = spark.read.parquet(BASE_PATH)

print('=== Final table (sorted by employee_id) ===')
final_df.orderBy('employee_id').show(25, truncate=False)

# Check 1: employee 10 is now in dept 35
emp10_final = final_df.filter(col('employee_id') == EMPLOYEE_ID).first()
assert emp10_final['dept_id'] == TARGET_DEPT, 'FAIL: employee 10 not in target dept'
print(f'CHECK 1 PASS: Employee {EMPLOYEE_ID} is now in dept_id = {emp10_final["dept_id"]}')

# Check 2: dept 20 and dept 30 are untouched (still 5 employees each)
for dept in [20, 30]:
    count = final_df.filter(col('dept_id') == dept).count()
    assert count == 5, f'FAIL: dept {dept} has {count} rows (expected 5)'
    print(f'CHECK 2 PASS: dept_id={dept} still has {count} employees (untouched)')

# Check 3: dept 10 now has 4 employees (employee 10 removed)
dept10_count = final_df.filter(col('dept_id') == 10).count()
assert dept10_count == 4, f'FAIL: dept 10 has {dept10_count} rows (expected 4)'
print(f'CHECK 3 PASS: dept_id=10 now has {dept10_count} employees (employee 10 removed)')

# Check 4: dept 35 now has 6 employees (employee 10 added)
dept35_count = final_df.filter(col('dept_id') == 35).count()
assert dept35_count == 6, f'FAIL: dept 35 has {dept35_count} rows (expected 6)'
print(f'CHECK 4 PASS: dept_id=35 now has {dept35_count} employees (employee 10 added)')

print('\nAll checks passed.')

## Step 8 — Inspect Partition Files on Disk

Confirm that only the two affected partitions have new (recently written) files,
while dept 20 and dept 30 directories remain from the original write.

In [ ]:
import time

print('Partition layout after dynamic overwrite:')
for entry in sorted(os.listdir(BASE_PATH)):
    if not entry.startswith('dept_id='):
        continue
    pdir = os.path.join(BASE_PATH, entry)
    files = sorted(f for f in os.listdir(pdir) if f.endswith('.parquet'))
    rows = final_df.filter(col('dept_id') == int(entry.split('=')[1])).count()
    print(f'  {entry}/ — {len(files)} file(s), {rows} rows')

print('\nKey takeaway:')
print('  dept_id=10 and dept_id=35 were rewritten (dynamic overwrite).')
print('  dept_id=20 and dept_id=30 were NOT touched — no read, no write.')

## Teardown

In [ ]:
import shutil

shutil.rmtree(BASE_PATH, ignore_errors=True)
print('Temp directory cleaned up.')

spark.stop()
print('SparkSession stopped.')